# 13.13 PPO裁切的是鼓勵，還是所有機率？


優勢+1的一張卡，應提高選擇機率；但ratio已從1升到1.3時，能否暫停同一筆舊評語的額外鼓勵？PPO的**裁切（clipping）**做的是這種限制，不是把新策略所有機率硬寫進上下限。

設區間0.8至1.2，每筆先算兩項：`ratio×advantage`，以及`clamp(ratio,0.8,1.2)×advantage`，取較小者作待提高的替代目標。clamp把超出區間的數壓回端點；最後取較小值這一步不可省。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
import torch
from tiny_perceptron.posttraining import ppo_clipped_objective

ratio = torch.tensor([0.7, 1.0, 1.3, 0.7, 1.0, 1.3], requires_grad=True)
advantage = torch.tensor([1.0, 1.0, 1.0, -1.0, -1.0, -1.0])
old_log_probability = torch.full((6,), 0.5).log()
new_log_probability = old_log_probability + ratio.log()
result = ppo_clipped_objective(new_log_probability, old_log_probability, advantage, clip_range=0.2)
loss = -result["surrogate"].sum()
loss.backward()
print("待提高的值", [round(value, 2) for value in result["surrogate"].tolist()])
print("下降代價時對ratio的梯度", [round(value, 2) for value in ratio.grad.tolist()])

前三格優勢+1，後三格−1，每組試ratio0.7、1、1.3。舊機率0.5，新機率0.35、0.5、0.65都合法。待提高的值是`[0.7,1,1.2,−0.8,−1,−1.3]`；取負作下降代價，對ratio的梯度是`[−1,−1,0,0,1,1]`。


In [ ]:
# @title 本節圖解
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/rewrite-13-clip-cases.svg")))

正優勢應增機率，超過1.2的一側變平；負優勢應減機率，低於0.8的一側變平。但負優勢若反而升到1.3，目標仍下降、梯度仍要把它拉回。只clamp後相乘，會把錯方向也隨意截平。

「變平」只表示此項暫停更多有利方向的鼓勵。其他樣本與共享權重仍可能改機率，所以ratio不是被保證留在區間；還要量實際偏移。

練習把clip_range改0.1，正組最後變1.1、負組第一變−0.9，其餘不變。這六個點的梯度也不變。

<details>
<summary>補充：來源、完整設定與既有實測紀錄</summary>

[PPO原論文公式7與圖1](https://arxiv.org/pdf/1707.06347)給出裁切替代目標與正負優勢的兩種形狀。

</details>
